# Format Data

In [5]:
import pandas as pd
from pathlib import Path
from omegaconf import OmegaConf
from olist_ml.config import OlistConfig
from sqlalchemy import create_engine

root_path = project_root=Path.cwd().parent.parent
cfg = OmegaConf.load("../../conf/config.yaml")

raw_data_path = root_path / cfg.paths.raw_data
olist_sqlite_path = raw_data_path / cfg.files.olist_sqlite
db_file = f"sqlite:///{olist_sqlite_path}"
engine = create_engine(db_file)
demand_by_category = pd.read_sql("""
    SELECT
        strftime('%Y-%m', o.order_purchase_timestamp) as order_purchase_date,
        pcn.product_category_name_english as category_name,
        SUM(oi.price + oi.freight_value) as total_price
    FROM order_items oi
    LEFT JOIN orders o ON o.order_id = oi.order_id
    LEFT JOIN products p ON oi.product_id = p.product_id
    LEFT JOIN product_category_name_translation pcn ON p.product_category_name = pcn.product_category_name
    WHERE p.product_category_name IN (
        SELECT
            p2.product_category_name
        FROM order_items oi2
        LEFT JOIN products p2 ON oi2.product_id = p2.product_id
        WHERE oi2.price <= 8000 
          AND oi2.freight_value <= 80
        GROUP BY p2.product_category_name
        HAVING COUNT(*) >= 3000
    )
    GROUP BY category_name, order_purchase_date
    ORDER BY order_purchase_date
""", engine)

In [9]:
furniture_decor_demand = demand_by_category[demand_by_category["category_name"] == "furniture_decor"]
telephony_demand = demand_by_category[demand_by_category["category_name"] == "telephony"]
health_beauty = demand_by_category[demand_by_category["category_name"] == "health_beauty"]

print(furniture_decor_demand.head())
print(telephony_demand.head())
print(health_beauty.head())

   order_purchase_date    category_name  total_price
0              2016-09  furniture_decor       136.23
7              2016-10  furniture_decor      7052.28
20             2017-01  furniture_decor     16461.75
33             2017-02  furniture_decor     26262.32
46             2017-03  furniture_decor     29506.43
   order_purchase_date category_name  total_price
2              2016-09     telephony        75.06
13             2016-10     telephony       633.98
26             2017-01     telephony      3411.51
39             2017-02     telephony      8647.34
52             2017-03     telephony     10731.96
   order_purchase_date  category_name  total_price
1              2016-09  health_beauty       143.46
9              2016-10  health_beauty      5493.38
22             2017-01  health_beauty     14039.74
35             2017-02  health_beauty     26130.08
48             2017-03  health_beauty     29852.87
